## Integrated Multi-Omics Data Visualization Platform

### 📌 Project Overview

In modern systems biology, analyzing a single data modality (like transcriptomics alone) often misses the phenotypic context provided by downstream translation. This project is a production-grade, interactive web platform designed to bridge the gap between **transcriptomics**, **proteomics**, and **pathway topology** in a unified dashboard interface.

By integrating multi-omics expression matrices dynamically, the platform enables researchers to cross-examine mRNA expression against corresponding protein abundances, calculate statistical correlation distributions, and visualize how these molecules interact within functional biological pathway networks.

### 🚀 Key Structural Modules

* **Parallel Modality Heatmaps:** Synchronized dual-matrix heatmaps that display side-by-side comparative expression profiles (e.g., Log2 CPM counts vs. Mass Spec intensities) across identical sample cohorts to identify immediate multi-layer expression trends.
* **Cross-Omics Feature Concordance:** An automated statistical layer that computes Pearson correlation coefficients ($r$) for matching features across datasets, isolating which genes maintain strict transcriptional-translational alignment versus those subject to post-transcriptional regulation.
* **Interactive Pathway Topology Networks:** A dynamic network visualization engine powered by `NetworkX` and `Plotly`. It models molecules as nodes and sets interactive edges based on user-defined correlation thresholds, displaying multi-omics mean values directly on the network nodes.

### 🛠️ Technical Architecture & Stack

* **Dashboard Framework:** `Streamlit` (Optimized with runtime caching flags for high-throughput data processing)
* **Interactive Graphics Engine:** `Plotly (Express & Graph Objects)`
* **Mathematical & Network Analytics:** `NetworkX`, `SciPy (Stats Module)`, `NumPy`, `Pandas`
* **Deployment Engineering:** Automated secure multi-threaded tunneling via `pyngrok` for rapid-prototype cloud hosting.


### 🎯 Portfolio Value Focus

* **Systems Biology Competency:** Demonstrates an advanced understanding of data integration challenges, moving beyond basic isolated data analysis.
* **Full-Stack Development:** Showcases the ability to build functional, interactive graphical user interfaces (GUIs) tailored specifically for research clinicians and bioinformaticians.
* **Advanced Network Modeling:** Explores network science principles by mapping statistical thresholds to physiological pathway structures.

In [1]:
print("⏳ Installing multi-omics analytics engines and system packages...")
!pip install -q streamlit plotly networkx scipy pandas numpy

# Install localtunnel to expose the web server port out of Google Colab safely
!npm install -g localtunnel -q

print("✅ All analytics dependencies installed successfully!")

⏳ Installing multi-omics analytics engines and system packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 56.8 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 22 packages in 2s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇✅ All analytics dependencies installed successfully!


In [2]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import networkx as nx
from scipy.stats import pearsonr

# Page styling matrix configuration
st.set_page_config(page_title="Multi-Omics Integration Platform", layout="wide")

st.title("🧬 Integrated Multi-Omics Data Visualization Platform")
st.markdown("Connect transcriptomic expressions, proteomic abundances, and biological pathway networks dynamically.")
st.markdown("---")

# Synthetic data generator for rapid environment testing
@st.cache_data
def generate_mock_omics_data():
    np.random.seed(42)
    genes = [f"GENE_{i:03d}" for i in range(1, 51)]
    samples = [f"Tumor_{i:02d}" for i in range(1, 6)] + [f"Normal_{i:02d}" for i in range(1, 6)]

    # Transcriptomics (Log2 CPM)
    rna_data = np.random.normal(loc=8, scale=2, size=(len(genes), len(samples)))
    rna_data[:10, :5] += 2.5 # Simulate dynamic upregulation variations in tumor tissue
    df_rna = pd.DataFrame(rna_data, index=genes, columns=samples)

    # Proteomics (Mass Spec Int)
    prot_data = rna_data * 0.7 + np.random.normal(loc=2, scale=1, size=(len(genes), len(samples)))
    df_prot = pd.DataFrame(prot_data, index=genes, columns=samples)

    # Target pathway configurations
    pathways = {
        "MAPK Signaling": genes[:15],
        "PI3K-Akt Pathway": genes[12:30],
        "Apoptosis": genes[28:40],
        "Cell Cycle": genes[35:50]
    }
    return df_rna, df_prot, pathways

df_rna, df_prot, pathway_dict = generate_mock_omics_data()

# Workspace interactive control handles
st.sidebar.header("⚙️ Workspace Controls")
selected_pathway = st.sidebar.selectbox("Select Functional Pathway Module", list(pathway_dict.keys()))
correlation_threshold = st.sidebar.slider("Cross-Omics Correlation Cutoff (r)", 0.0, 1.0, 0.5, 0.05)

pathway_genes = pathway_dict[selected_pathway]
filtered_rna = df_rna.loc[df_rna.index.isin(pathway_genes)]
filtered_prot = df_prot.loc[df_prot.index.isin(pathway_genes)]
st.sidebar.success(f"Loaded {len(pathway_genes)} target components within {selected_pathway}.")

# MODULE 1: HEATMAP EXPRESSION LAYOUTS
st.header("📊 1. Parallel Expression Profiles (Transcript vs. Protein)")
col1, col2 = st.columns(2)

with col1:
    st.subheader("Transcriptomics (mRNA Expression)")
    fig_rna = px.imshow(filtered_rna, labels=dict(x="Samples", y="Genes", color="Log2 CPM"), color_continuous_scale="RdBu_r")
    st.plotly_chart(fig_rna, use_container_width=True)

with col2:
    st.subheader("Proteomics (Protein Abundance)")
    fig_prot = px.imshow(filtered_prot, labels=dict(x="Samples", y="Genes", color="Intensity"), color_continuous_scale="Viridis")
    st.plotly_chart(fig_prot, use_container_width=True)

# MODULE 2: CORRELATION CONCORDANCE AUDIT
st.header("📈 2. Cross-Omics Feature Concordance")
correlation_results = []
for gene in pathway_genes:
    if gene in df_rna.index and gene in df_prot.index:
        r_val, _ = pearsonr(df_rna.loc[gene], df_prot.loc[gene])
        correlation_results.append({"Gene": gene, "Pearson_r": r_val})

df_corr = pd.DataFrame(correlation_results).sort_values(by="Pearson_r", ascending=False)
col_chart, col_data = st.columns([2, 1])

with col_chart:
    fig_bar = px.bar(df_corr, x="Gene", y="Pearson_r", color="Pearson_r", color_continuous_scale="Tropic")
    st.plotly_chart(fig_bar, use_container_width=True)
with col_data:
    st.subheader("Audit Records")
    st.dataframe(df_corr, use_container_width=True, height=280)

# MODULE 3: INTERACTIVE NETWORK VISUALIZATION
st.header("🕸️ 3. Multi-Omics Pathway Topology Network")
G = nx.Graph()
for gene in pathway_genes:
    G.add_node(gene, rna=df_rna.loc[gene].mean(), prot=df_prot.loc[gene].mean())

for idx, row in df_corr.iterrows():
    if abs(row["Pearson_r"]) >= correlation_threshold:
        current_gene = row["Gene"]
        other_genes = df_corr[df_corr["Gene"] != current_gene]["Gene"].head(2).values
        for target in other_genes:
            G.add_edge(current_gene, target, weight=row["Pearson_r"])

pos = nx.spring_layout(G, seed=42)
edge_x, edge_y = [], []
for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

edge_trace = go.Scatter(x=edge_x, y=edge_y, line=dict(width=1.5, color='#888'), hoverinfo='none', mode='lines')
node_x, node_y, node_hover, node_color = [], [], [], []

for node in G.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)
    node_color.append(G.nodes[node]['rna'])
    node_hover.append(f"🧬 Symbol: {node}<br>Avg RNA: {G.nodes[node]['rna']:.2f}<br>Avg Prot: {G.nodes[node]['prot']:.2f}")

node_trace = go.Scatter(
    x=node_x, y=node_y, mode='markers+text', text=list(G.nodes()), textposition="top center", hoverinfo='text', hovertext=node_hover,
    marker=dict(showscale=True, colorscale='YlGnBu', color=node_color, size=24, line_width=2, colorbar=dict(title="Avg RNA Log2 CPM"))
)

fig_network = go.Figure(data=[edge_trace, node_trace], layout=go.Layout(showlegend=False, xaxis=dict(showgrid=False, zeroline=False, showticklabels=False), yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)))
st.plotly_chart(fig_network, use_container_width=True)

Writing app.py


In [5]:
import subprocess
import time
import sys

# 1. Automatic installation check to prevent ModuleNotFoundError
try:
    from pyngrok import ngrok
except ModuleNotFoundError:
    print("📦 'pyngrok' package not found in current session. Installing it now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyngrok"])
    from pyngrok import ngrok
    print("✅ 'pyngrok' installed successfully!")

# 2. Paste your free ngrok Authtoken here to authenticate your cloud workspace safely
# Get your token from: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "3EMUT2tRQi9STwsmgFQGKMw3fK1_4yKb6mGiaepAQUbhMX64B"
ngrok.set_auth_token(NGROK_TOKEN)

print("⏳ Launching Streamlit core platform process...")
# 3. Spin up the Streamlit background data app
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

# 4. Critical Safety Delay: Wait 3 seconds to let the server bind to the port completely
time.sleep(3)

print("🚀 Establishing stable network tunnel endpoint...")
try:
    # Kill any lingering old connections to free up the socket channel cleanly
    ngrok.kill()

    # Connect a fresh, stable public tunnel hook
    public_url = ngrok.connect(8501, proto="http")

    print("\n" + "="*70)
    print("🎯 ACTIVE PLATFORM LINK:")
    print(f"👉 {public_url.public_url} 👈")
    print("="*70)
    print("Click the URL above to open your dashboard cleanly!")
except Exception as e:
    print(f"\n❌ Tunnel Connection Error: {e}")
    print("\n💡 Troubleshooting Tips:")
    print("1. Double-check that your NGROK_TOKEN string is pasted correctly.")
    print("2. If the token is correct and it still errors, go to 'Runtime' -> 'Restart session' in Colab and run it again.")

⏳ Launching Streamlit core platform process...
🚀 Establishing stable network tunnel endpoint...

🎯 ACTIVE PLATFORM LINK:
👉 https://hermit-badness-kennel.ngrok-free.dev 👈
Click the URL above to open your dashboard cleanly!
